## LVM — flexible volume management

**LVM (Logical Volume Manager)** sits between partitions and filesystems. It pools multiple disks or partitions together, then carves **logical volumes** out of the pool — and lets you **resize them on the fly**, without losing data. Three layers:

```
Physical Volumes (PVs)   /dev/sdb1, /dev/sdc1   real disks/partitions
      ↓ aggregated into
Volume Group (VG)        "datavg"                one big pool
      ↓ carved into
Logical Volumes (LVs)    "appdata", "logs"       act like partitions
      ↓ formatted with
Filesystems              ext4 / xfs on each LV
```

**Create it bottom-up:**

```bash
sudo pvcreate /dev/sdb1 /dev/sdc1        # 1. mark disks as PVs
sudo vgcreate datavg /dev/sdb1 /dev/sdc1 # 2. pool them into a VG
sudo lvcreate -L 20G -n appdata datavg   # 3. carve out a 20 GB LV
sudo mkfs.ext4 /dev/datavg/appdata       # 4. format, then mount
```

Inspect each layer with `pvs` / `vgs` / `lvs` (or the `…display` verbose forms).

**The superpower is growing an LV live**, filesystem and all, with one command:

```bash
sudo lvextend -L +5G -r /dev/datavg/appdata     # +5G, -r grows the FS too
```

The **`-r`** flag resizes the underlying `ext4`/`xfs` in the same call. Out of pool space? Add a disk: `pvcreate` it, then **`vgextend datavg /dev/sdd1`**. Shrinking works only on ext4 (XFS can't) and needs an unmount + `e2fsck` first — **growing is the LVM win; do that whenever you can.** LVM also does instant **snapshots** (`lvcreate -s`) for consistent backups.
